In [6]:
# ==============================================================================
# CELL 1: IMPORTS & ENVIRONMENTAL GENERATION (WAVES AND WIND)
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import root_scalar
from scipy.fft import fft, fftfreq
import scipy.integrate as scp
import pandas as pd
import time
import os

start_time = time.time()

# --- WAVE & WIND FUNCTIONS ---
def lcg(seed, a=1103515245, c=12345, m=2**31, n=1):
    """Linear congruential random number generator (matches MATLAB)."""
    numbers = np.ones(n) * seed
    for i in range(1, n):
        numbers[i] = (a * numbers[i-1] + c) % m
    return 2*np.pi*np.array([x / m for x in numbers])  # Normalize to [0, 1]

def calculateJONSWAPSpectrum(waveDict):
    Hs, Tp = waveDict["Hs"], waveDict["Tp"]
    gamma = waveDict.get("gamma", 1.0)
    df = waveDict["TDur"]**-1.0
    f = np.arange(df, waveDict["fHighCut"] + df, df)
    sigma = np.ones(len(f))
    fp = 1./Tp
    sigma[f>fp] = 0.09
    sigma[f<=fp] = 0.07        
    Spectrum = 0.3125 * Hs**2 * Tp * (f / fp)**(-5) * np.exp(-1.25 * (f / fp)**(-4)) * \
               (1 - 0.287 * np.log(gamma)) * gamma**(np.exp(-0.5 * (((f/fp) - 1) / sigma)**2)) 
    
    out = dict(waveDict)
    out.update({"Spectrum": Spectrum, "amplitudeSpectrum": np.sqrt(2*Spectrum*df), "f": f})
    return out

def generateRandomPhases(inputDict, seed=2):
    phi = lcg(seed, n=len(inputDict["Spectrum"]))
    out = dict(inputDict)
    out["randomPhases"] = phi
    return out

def dispersion(f, h):
    omega = 2*np.pi*f
    k = np.zeros_like(f)
    g = 9.81
    myFun = lambda k, omega, g, h: omega**2 - g*k*np.tanh(k*h)
    myFunPrime = lambda k, omega, g, h: -g*(k*h*(1-np.tanh(k*h)**2)+np.tanh(k*h))
    kGuess = omega[0] / np.sqrt(g*h)
    for j in range(len(f)):
        k[j] = root_scalar(lambda x: myFun(x,omega[j],g,h), fprime=lambda x: myFunPrime(x,omega[j], g,h), x0=kGuess, method='newton').root
        kGuess = k[j]
    return k

def pad2(vector, size):
    return np.pad(vector, [1, size - len(vector) - 1])

def calculateFreeSurfaceElevationTimeSeriesFFT(waveDict):
    t = waveDict["t"]
    M = len(t)
    kernel = waveDict["amplitudeSpectrum"] * np.exp(1j * waveDict["randomPhases"])
    eta = np.fft.ifft(pad2(kernel, M)).real * M
    out = dict(waveDict)
    out.update({"t": t, "eta": eta})
    return out

def calculateKinematicsFFT(inputDict):
    t, f, h, z = inputDict["t"], inputDict["f"], inputDict["h"], inputDict["z"]
    omega = 2*np.pi*f
    u, ut, phi = np.zeros((len(t), len(z))), np.zeros((len(t), len(z))), np.zeros((len(t), len(z)))
    k = dispersion(f, h)
    M = len(t)
    
    for j_, z_ in enumerate(z):
        cosh_term = np.cosh(k*(z_ + h)) / np.sinh(k * h)
        uKernel = inputDict["amplitudeSpectrum"] * omega * cosh_term * np.exp(1j*inputDict["randomPhases"])
        u[:, j_] = np.fft.ifft(pad2(uKernel, M)).real * M
        utKernel = inputDict["amplitudeSpectrum"] * omega**2 * cosh_term * np.exp(1j*(inputDict["randomPhases"] + (np.pi/2)))
        ut[:, j_] = np.fft.ifft(pad2(utKernel, M)).real * M
        pKernel = -1025.0 * inputDict["amplitudeSpectrum"] * 9.81 * cosh_term * np.exp(1j*inputDict["randomPhases"])
        phi[:, j_] = np.fft.ifft(pad2(pKernel, M)).real * M
        
    out = dict(inputDict)
    out.update({"u": u, "ut": ut, "phi": phi})
    return out

def calculateKaimalSpectrum(windDict):
    df = windDict["TDur"]**-1
    f = np.arange(df, windDict["fHighCut"], df)
    Spectrum = (4 * windDict["I"]**2 * windDict["V_10"] * windDict["l"]) / (1 + 6 * ((f * windDict["l"])/windDict["V_10"]))**(5/3)
    out = dict(windDict)
    out.update({"Spectrum": Spectrum, "amplitudeSpectrum": np.sqrt(2*Spectrum*df), "f": f})
    return out

def calculateWindTimeSeriesFFT(windDict):
    t = windDict["t"]
    M = len(t)
    kernel = windDict["amplitudeSpectrum"] * np.exp(1j * windDict["randomPhases"])
    V_hub = np.fft.ifft(pad2(kernel, M)).real * M + windDict["V_10"]
    out = dict(windDict)
    out.update({"t": t, "V_hub": V_hub})
    return out

# --- DICTIONARIES & GENERATION ---
timeInfo = {"TTrans":50, "TDur":501, "dt":0.5, "fHighCut":0.5}
wind = {"I": 0.14, "l": 340.2, "V_10": 8.0}
wave = {"Hs": 7.5, "Tp": 11, "gamma": 1.0, "h": 320, "z": np.arange(-320, 1, 1)}

# Generate Waves
wave.update(timeInfo)
waves = calculateKinematicsFFT(calculateFreeSurfaceElevationTimeSeriesFFT(generateRandomPhases(calculateJONSWAPSpectrum(wave), seed=2)))

# Generate Wind
wind.update(timeInfo)
wind["t"] = waves["t"]
wind = calculateWindTimeSeriesFFT(generateRandomPhases(calculateKaimalSpectrum(wind)))

# Plotting Environments
fig, axs = plt.subplots(1, 2, figsize=(15, 4))
axs[0].plot(waves["t"], waves["eta"])
axs[0].set_title("Irregular Wave Elevation")
axs[0].grid()
axs[1].plot(wind["t"], wind["V_hub"], color='orange')
axs[1].set_title("Wind Speed at Hub")
axs[1].grid()
plt.show()

KeyError: 't'

In [3]:
# ==============================================================================
# CELL 2: MOORING & THRUST LOOKUP TABLES
# ==============================================================================
# Mooring Table [offset -> Fxl, Fzl, Fxr, Fzr]
mooring_offset = np.array([0, 3.49, 7.28, 10.15, 12.31, 13.9, 15.12, 16.23, 17.15, 17.98, 18.74, 19.5, 20.22, 20.89, 21.54, 22.19, 22.85, 23.5, 24.13, 24.75, 25.38, 25.97, 26.59, 27.22, 27.84, 28.44, 29.05, 29.67, 30.25, 30.86, 31.48])
mooring_table = np.array([
    [-301446.60, -349797.48, -410631.37, -470825.79, -556149.09, -645566.18, -734675.91, -833331.58, -927824.59, -1022160.27, -1115123.41, -1213381.04, -1310492.89, -1403732.66, -1496351.12, -1590714.92, -1687995.05, -1784986.08, -1879919.14, -1974086.30, -2070403.15, -2161091.87, -2256823.13, -2354484.90, -2450921.91, -2544512.82, -2639895.02, -2737049.16, -2828101.84, -2924017.01, -3021646.51],
    [-2687887.68, -3004799.94, -3527369.94, -4198179.18, -4958975.40, -5756274.47, -6550832.95, -7430509.03, -8273068.15, -9114224.45, -9943142.33, -10819269.21, -11685179.61, -12516564.08, -13342408.64, -14183815.66, -15051226.55, -15916059.56, -16762542.50, -17602196.23, -18461017.86, -19269655.60, -20123255.79, -20994069.56, -21853962.69, -22688478.16, -23538965.89, -24405253.32, -25217136.37, -26072376.43, -26942902.48],
    [315260.63, 240340.22, 210548.57, 185368.97, 164951.50, 158680.01, 146983.87, 145386.55, 140036.26, 131021.36, 128771.53, 128483.45, 126479.43, 122553.74, 115161.93, 113569.01, 113591.64, 112059.31, 108700.09, 107329.48, 106005.16, 99718.59, 98526.42, 95561.12, 94435.99, 94568.53, 93484.57, 92433.89, 86963.23, 86009.09, 83442.96],
    [-2693198.75, -2396810.98, -2140844.16, -1975535.70, -1865967.23, -1789537.39, -1736371.80, -1688767.71, -1651371.05, -1619547.19, -1591737.18, -1562691.06, -1538317.03, -1514335.98, -1492701.74, -1472054.66, -1449827.49, -1430269.62, -1410625.81, -1392839.12, -1375653.03, -1358563.91, -1342321.89, -1324478.49, -1308884.19, -1292147.63, -1277336.79, -1262980.69, -1248574.86, -1234875.69, -1219926.43]
])
mooring_table[1] -= mooring_table[1][0]
mooring_table[3] -= mooring_table[3][0]

def Fmoor(x):
    fxl = np.interp(x, mooring_offset, mooring_table[0])
    fzl = np.interp(x, mooring_offset, mooring_table[1])
    fxr = np.interp(x, mooring_offset, mooring_table[2])
    fzr = np.interp(x, mooring_offset, mooring_table[3])
    return fxl, fzl, fxr, fzr

# Thrust Table
thrust_table = {
    "x": [0, 3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24],
    "y": [0, 173.91, 217.61, 284.44, 348.28, 395.25, 479.78, 589.49, 702.76, 807.76, 692.91, 611.48, 543.47, 521.73, 499.71, 456.31, 435.26, 434.44, 413.38, 391.58, 391.30, 390.11, 349.40]
}

In [4]:
# ==============================================================================
# CELL 3: STRUCTURAL PROPERTIES (TOWER, RNA, BUOY)
# ==============================================================================
# Basic Constants
E, G = 210e9, 80.8e9   # [Pa]
rho, rho_water = 8500, 1025 # [kg/m^3]
g = 9.81
Fcurrent_avg = 1797

# --- TOWER PROPERTIES ---
elev_tower = np.array([10.0, 17.76, 25.52, 33.28, 41.04, 48.80, 56.56, 64.32, 72.08, 79.84, 87.60])
rA = np.array([4667.0, 4345.28, 4034.76, 3735.44, 3447.32, 3170.40, 2904.69, 2650.18, 2406.88, 2174.77, 1953.87])
EI = np.array([603.9e9, 517.6e9, 440.9e9, 373.0e9, 313.2e9, 260.8e9, 215.3e9, 176.0e9, 142.3e9, 113.6e9, 89.4e9])
EA = np.array([115.3e9, 107.3e9, 99.6e9, 92.2e9, 85.1e9, 78.3e9, 71.7e9, 65.4e9, 59.4e9, 53.7e9, 48.2e9])
rI = np.array([24443.7, 20952.2, 17847.0, 15098.5, 12678.6, 10560.1, 8717.2, 7124.9, 5759.8, 4599.3, 3622.1])

D_base, t_base = 6.5, 0.027
D_top, t_top = 3.87, 0.019
nel_tower = 15
nn_tower = nel_tower + 1
z_base, z_top = 10, 87.6

z_tower = np.linspace(z_base, z_top, nel_tower)
zn_tower = np.linspace(z_base, z_top, nn_tower)

rA_tower = np.interp(z_tower, elev_tower, rA)
EI_tower = np.interp(z_tower, elev_tower, EI)
EA_tower = np.interp(z_tower, elev_tower, EA)
rI_tower = np.interp(z_tower, elev_tower, rI)
D_tower = np.linspace(D_base, D_top, nel_tower)
R_tower = D_tower / 2
rAw_tower = rho_water * np.pi * R_tower**2

# --- RNA PROPERTIES ---
H_hub, u_offset = 90, 5
m_hub, m_blade, m_nacelle = 56780, 17740, 240000
I_blade = 11776047

m_RNA = m_hub + m_nacelle + 3*m_blade
cog_hub = np.array([u_offset, 0, H_hub])
cog_blade = np.array([u_offset, 0, H_hub])
cog_nacelle = np.array([-1.9, 0, H_hub])
cog_RNA = (m_hub*cog_hub + m_nacelle*cog_nacelle + 3*m_blade*cog_blade) / m_RNA
I55_RNA = m_RNA*(cog_RNA[0]**2 + (H_hub - z_top)**2) + 2*I_blade

# --- BUOY PROPERTIES ---
draft, z_top_buoy = -120, 10
z_top_taper_buoy, z_base_taper_buoy = -4, -12
D_above_buoy, D_below_buoy = 6.5, 9.4
H_taper_buoy = np.abs(z_top_taper_buoy - z_base_taper_buoy)
taper_slope_buoy = (D_below_buoy - D_above_buoy) / H_taper_buoy

m_buoy = 7466330
cog_buoy = -89.9155

zn_range = np.array([draft, z_base_taper_buoy, z_top_taper_buoy, 0, z_top_buoy])
R_buoy_key = np.array([D_below_buoy/2, D_below_buoy/2, D_above_buoy/2, D_above_buoy/2, D_above_buoy/2])
t_buoy_key = np.ones_like(zn_range)*0.0365

A_buoy_key = np.pi*(R_buoy_key**2 - (R_buoy_key-t_buoy_key)**2)
V0_buoy = np.trapz(np.pi*R_buoy_key**2, zn_range) - np.pi*R_buoy_key[-1]**2*z_top_buoy

rA_buoy_key = rho * A_buoy_key
CM_shell = np.trapz(rA_buoy_key*zn_range, zn_range) / np.trapz(rA_buoy_key, zn_range)

m_shell = np.trapz(rA_buoy_key, zn_range)
m_total = m_buoy - m_RNA
m_ballast = m_total - m_shell
CM_ballast = (cog_buoy*m_total - m_shell*CM_shell) / m_ballast
H_ballast = 2*(CM_ballast - zn_range[0])
V_ballast = H_ballast*np.pi*(R_buoy_key[0]-t_buoy_key[0])**2
rho_ballast = m_ballast / V_ballast
rA_ballast_val = rho_ballast*np.pi*(R_buoy_key[0]-t_buoy_key[0])**2

In [5]:
# ==============================================================================
# CELL 4: DISCRETIZATION AND PROPERTY MAPPING
# ==============================================================================
def interpolate_nodes_preserving_keys(zn_range, total_nodes):
    total_segments = total_nodes - 1
    seg_lengths = np.diff(zn_range)
    raw_segments = seg_lengths / np.sum(seg_lengths) * total_segments
    seg_counts = np.round(raw_segments).astype(int)
    
    while np.sum(seg_counts) != total_segments:
        diff = total_segments - np.sum(seg_counts)
        idx = np.argmax(raw_segments - seg_counts if diff > 0 else seg_counts - raw_segments)
        seg_counts[idx] += np.sign(diff)

    zn = []
    for i in range(len(zn_range) - 1):
        zn.extend(np.linspace(zn_range[i], zn_range[i+1], seg_counts[i] + 1, endpoint=False))
    zn.append(zn_range[-1])
    return np.array(zn)

nel_buoy = 15
nn_buoy = nel_buoy + 1
zn_range_fill = [draft, draft+H_ballast, z_base_taper_buoy, z_top_taper_buoy, 0, z_top_buoy]
zn_buoy = interpolate_nodes_preserving_keys(zn_range_fill, nn_buoy)
nel_buoy = len(zn_buoy) - 1

R_buoy_interp, rA_buoy_interp, rI_buoy_interp = np.zeros(nel_buoy), np.zeros(nel_buoy), np.zeros(nel_buoy)
t_buoy = np.ones(nel_buoy)*0.0365

for i, z in enumerate((zn_buoy[:-1] + zn_buoy[1:])/2):
    if z < draft + H_ballast:
        rA_buoy_interp[i] = rA_buoy_key[0] + rA_ballast_val
        R_buoy_interp[i] = R_buoy_key[0]
        rI_buoy_interp[i] = rho*np.pi/4*(R_buoy_key[0]**4 - (R_buoy_key[0]-0.0365)**4) + np.pi/4*rho_ballast*(R_buoy_key[0] - 0.0365)**4
    elif z <= z_base_taper_buoy:
        rA_buoy_interp[i], R_buoy_interp[i] = rA_buoy_key[1], R_buoy_key[1]
        rI_buoy_interp[i] = rho*np.pi/4*(R_buoy_key[1]**4 - (R_buoy_key[1]-0.0365)**4)
    elif z <= z_top_taper_buoy:
        R_buoy_interp[i] = (D_below_buoy - taper_slope_buoy * (z - z_base_taper_buoy)) / 2
        rA_buoy_interp[i] = np.interp(z, zn_range_fill[1:], rA_buoy_key)
        rI_buoy_interp[i] = rho*np.pi/4*(R_buoy_interp[i]**4 - (R_buoy_interp[i]-0.0365)**4)
    else:
        R_buoy_interp[i] = D_above_buoy / 2
        rA_buoy_interp[i] = rA_buoy_key[2]
        rI_buoy_interp[i] = rho*np.pi/4*(R_buoy_key[-1]**4 - (R_buoy_key[-1]-0.0365)**4)

R_buoy, rA_buoy, rI_buoy = R_buoy_interp, rA_buoy_interp, rI_buoy_interp
A_buoy = np.pi*(R_buoy**2 - (R_buoy - t_buoy)**2)
I_buoy = np.pi/4*(R_buoy**4 - (R_buoy - t_buoy)**4)
EA_buoy, EI_buoy = E*A_buoy, E*I_buoy
rAw_buoy = rho_water*np.pi*R_buoy**2

# Combine Buoy and Tower
if np.isclose(zn_buoy[-1], zn_tower[0]):
    zn = np.append(zn_buoy[:-1], zn_tower)
else:
    zn = np.append(zn_buoy, zn_tower)

ne, nn = len(zn)-1, len(zn)
Fair_idx = np.argmin(np.abs(zn - (-70)))

elem_nodes = [[ie, ie+1] for ie in range(ne)]
elem_dofs = [np.arange(3*ie, 3*ie+6) for ie in range(ne)]
dof_node = [int(np.floor(idof / 3)) for idof in range(3*nn)]

rA, rI, rAw = np.append(rA_buoy, rA_tower), np.append(rI_buoy, rI_tower), np.append(rAw_buoy, rAw_tower)
EI, EA, Rs = np.append(EI_buoy, EI_tower), np.append(EA_buoy, EA_tower), np.append(R_buoy, R_tower)

In [ ]:
# ==============================================================================
# CELL 5: FINITE ELEMENT SHAPE FUNCTIONS
# ==============================================================================
h = np.diff(zn)
N_rod, dN_rod = [], []

for ie in range(ne):
    h_local = h[ie]
    ze = zn[elem_nodes[ie]]
    N_rod.append([lambda z, ze=ze, h=h_local: (ze[1] - z) / h, lambda z, ze=ze, h=h_local: (z - ze[0]) / h])
    dN_rod.append([lambda z, h=h_local: -1 / h + 0.0 * z, lambda z, h=h_local: 1 / h + 0.0 * z])

def base(z, h_loc): return np.array([[1], [z], [z**2], [z**3]]) if isinstance(z, float) else np.array([[np.ones(len(z))], [z], [z**2], [z**3]])
def dbase(z, h_loc): return np.array([[0], [1], [2 * z], [3 * z**2]]) if isinstance(z, float) else np.array([[np.zeros(len(z))], [np.ones(len(z))], [2 * z], [3 * z**2]])
def ddbase(z, h_loc): return np.array([[0], [0], [2], [6 * z]]) if isinstance(z, float) else np.array([[np.zeros(len(z))], [np.zeros(len(z))], [2*np.ones(len(z))], [6 * z]])

def make_N(coeff, h_loc): return lambda z: np.dot(np.transpose(base(z, h_loc)), coeff)
def make_dN(coeff, h_loc): return lambda z: np.dot(np.transpose(dbase(z, h_loc)), coeff)
def make_ddN(coeff, h_loc): return lambda z: np.dot(np.transpose(ddbase(z, h_loc)), coeff)

N_k_b, dN_k_b, ddN_k_b = [], [], []
for ie in range(ne):
    h_loc = h[ie]
    matrix = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [1, h_loc, h_loc**2, h_loc**3], [0, 1, 2 * h_loc, 3 * h_loc**2]])
    N_k_el, dN_k_el, ddN_k_el = [], [], []
    for idof in range(4):
        rhs = np.zeros(4); rhs[idof] = 1
        coeff = np.linalg.solve(matrix, rhs)
        N_k_el.append(make_N(coeff, h_loc))
        dN_k_el.append(make_dN(coeff, h_loc))
        ddN_k_el.append(make_ddN(coeff, h_loc))
    N_k_b.append(N_k_el)
    dN_k_b.append(dN_k_el)
    ddN_k_b.append(ddN_k_el)

In [ ]:
# ==============================================================================
# CELL 6: GLOBAL MATRIX ASSEMBLY
# ==============================================================================
Ca, Cd = 0.969954, 0.6
M_k_rod, K_k_rod = np.zeros((ne, 6, 6)), np.zeros((ne, 6, 6))
M_k_beam, K_k_beam, F_k_beam = np.zeros((ne, 6, 6)), np.zeros((ne, 6, 6)), np.zeros((ne, 6, 6))
R_k_beam, D_k_beam, I_k_beam = np.zeros((ne, 6, 6)), np.zeros((ne, 6, 6)), np.zeros((ne, 6, 6))
Q2cur_k = np.zeros((ne, 6))

beam_dofs, rod_dofs = {0: 0, 2: 1, 3: 2, 5: 3}, {1: 0, 4: 1}

# Weight Tracking
T_self = np.zeros(len(zn))
elem_weight = [rA[ie] * h[ie] for ie in range(ne)]
for i, z_node in enumerate(zn):
    for ie in range(ne):
        z_elem = max(zn[elem_nodes[ie]])
        if max(elem_nodes[ie]) == max(max(elem_nodes)): T_self[i] += m_RNA
        if z_elem > z_node: T_self[i] += elem_weight[ie]
T_self *= g

# Element Integration
for ie in range(ne):
    ze = zn[elem_nodes[ie]]
    L_loc = ze[1] - ze[0]
    for idof in range(6):
        for jdof in range(6):
            if idof in beam_dofs and jdof in beam_dofs:
                beam_i, beam_j = beam_dofs[idof], beam_dofs[jdof]
                if zn[ie + 1] <= 0: Q2cur_k[ie, idof] = scp.quad(lambda z: Fcurrent_avg*N_k_b[ie][beam_i](z), 0, L_loc)[0]
                
                M_k_beam[ie, idof, jdof] = scp.quad(lambda z: rA[ie] * N_k_b[ie][beam_i](z) * N_k_b[ie][beam_j](z), 0, L_loc)[0]
                K_k_beam[ie, idof, jdof] = scp.quad(lambda z: EI[ie] * ddN_k_b[ie][beam_i](z) * ddN_k_b[ie][beam_j](z) + T_self[ie] * dN_k_b[ie][beam_j](z) * dN_k_b[ie][beam_i](z), 0, L_loc)[0]
                F_k_beam[ie, idof, jdof] = scp.quad(lambda z: rA[ie] * g * dN_k_b[ie][beam_j](z) * N_k_b[ie][beam_i](z), 0, L_loc)[0]
                
                if zn[ie + 1] <= 0:
                    R_k_beam[ie, idof, jdof] = scp.quad(lambda z: rho_water * g * Rs[ie]**2 * np.pi * dN_k_b[ie][beam_j](z) * N_k_b[ie][beam_i](z), 0, L_loc)[0]
                    D_k_beam[ie, idof, jdof] = scp.quad(lambda z: 0.5 * rho_water * Cd * np.pi * Rs[ie]**2 * N_k_b[ie][beam_j](z) * N_k_b[ie][beam_i](z), 0, L_loc)[0]
                    I_k_beam[ie, idof, jdof] = scp.quad(lambda z: rho_water * (1 + Ca) * np.pi * Rs[ie]**2 * N_k_b[ie][beam_j](z) * N_k_b[ie][beam_i](z), 0, L_loc)[0]
                    
            if idof in [1, 4] and jdof in [1, 4]:
                rod_i, rod_j = (0 if idof==1 else 1), (0 if jdof==1 else 1)
                M_k_rod[ie, idof, jdof] = scp.quad(lambda z: rA[ie] * N_rod[ie][rod_i](z) * N_rod[ie][rod_j](z), ze[0], ze[1])[0]
                K_k_rod[ie, idof, jdof] = scp.quad(lambda z: EA[ie] * dN_rod[ie][rod_i](z) * dN_rod[ie][rod_j](z), ze[0], ze[1])[0]

# Add RNA & Buoyancy Boundary Conditions
M_k_beam[-1][3,3] += m_RNA
M_k_beam[-1][5,5] += I55_RNA
M_k_rod[-1][4,4] += m_RNA
D_k_beam[0][1,1] += 130000

C33 = rho_water*np.pi*R_buoy[-1]**2*g
K_k_rod[0][1,1] += C33
F_k_beam[-1][3,5] -= m_RNA*g
F_k_beam[-1][5,3] += m_RNA*g

# Assembly
ndofs = nn*3
K, M, Q, H, D, I_mat, L_cur = np.zeros((ndofs, ndofs)), np.zeros((ndofs, ndofs)), np.zeros((ndofs, ndofs)), np.zeros((ndofs, ndofs)), np.zeros((ndofs, ndofs)), np.zeros((ndofs, ndofs)), np.zeros(ndofs)

for ie in range(ne):
    dofs = elem_dofs[ie]
    nodes = np.append(3*dof_node[dofs[0]] + np.arange(3), 3*dof_node[dofs[-1]] + np.arange(3))
    for i in range(6):
        for j in range(6):
            M[nodes[i], nodes[j]] += M_k_rod[ie, i, j] + M_k_beam[ie, i, j]
            K[nodes[i], nodes[j]] += K_k_rod[ie, i, j] + K_k_beam[ie, i, j] 
            Q[nodes[i], nodes[j]] += F_k_beam[ie, i, j]
            H[nodes[i], nodes[j]] += R_k_beam[ie, i, j]
            D[nodes[i], nodes[j]] += D_k_beam[ie, i, j]
            I_mat[nodes[i], nodes[j]] += I_k_beam[ie, i, j]
            if i == 0 and (j == 0 or j == 3): L_cur[nodes[i]] += Q2cur_k[ie, j]

C = 0.01*M + 0.01*K
K += -Q + H
M += I_mat

In [ ]:
# ==============================================================================
# CELL 7: HYDROSTATIC PROPERTIES
# ==============================================================================
cob_buoy_check = np.trapz(np.pi*R_buoy_key[0:3]**2*zn_range[0:3], zn_range[0:3]) / np.trapz(np.pi*R_buoy_key[0:3]**2, zn_range[0:3])

Iy = np.pi*R_buoy[-1]**4/4
BM = Iy/V0_buoy
KB = cob_buoy_check - draft
KG = cog_buoy - draft
GM = KB + BM - KG

C55 = rho_water*g*V0_buoy*GM
I55 = 4229230000 + I55_RNA + m_RNA*(90-cog_buoy)**2
f5 = np.sqrt(C55/I55) / (2*np.pi)
f3 = np.sqrt(C33/m_buoy) / (2*np.pi)

print(f"BM = {BM:.3f} m, KB = {KB:.3f} m, KG = {KG:.3f} m, GM = {GM:.3f} m")
print(f"Pitch Natural Period = {1/f5:.2f} s")
print(f"Heave Natural Period = {1/f3:.2f} s")

In [ ]:
# ==============================================================================
# CELL 8: FULL FEM TIME DOMAIN SOLVE
# ==============================================================================
udofs, vdofs = np.arange(0, ndofs), np.arange(ndofs, 2*ndofs)
q0 = np.zeros(2*ndofs)

tf = timeInfo['TDur'] - 1
tspan = np.arange(0, tf, 0.05)

zn_sub = zn[zn <= 0]
nodal_uwave = np.array([[np.interp(z, waves['z'], waves['u'][i]) for z in zn_sub] for i in range(len(waves['t']))])

def uwave(t): return np.array([np.interp(t, waves['t'], nodal_uwave[:, i]) for i in range(len(zn_sub))])
def pbase(t): return np.interp(t, waves['t'], [np.interp(draft, waves['z'], waves['phi'][i]) for i in range(len(waves['t']))])
def V_hub(t): return np.interp(t, wind['t'], wind['V_hub'])

# Pre-invert M for speed
M_inv = np.linalg.inv(M)

Fz, Fw, Fm = np.zeros(ndofs), np.zeros(ndofs), np.zeros(ndofs)
idx = vdofs[0::3][0:len(zn_sub)]
qwave = np.zeros(2*ndofs)

def odefun(t, q):
    qwave[idx] = uwave(t)
    urel = q[vdofs] - qwave[vdofs]
    vrel = q[vdofs[-3]] - V_hub(t)
    
    Fz[1] = np.pi*R_buoy_key[0]**2 * pbase(t)
    Fw[-3] = np.interp(vrel, thrust_table['x'], thrust_table['y'])*1e3
    
    Fmxl, Fmzl, Fmxr, Fmzr = Fmoor(q[udofs][6])
    Fm[Fair_idx*3] = Fmxr + Fmxl
    Fm[Fair_idx*3+1] = Fmzl + Fmzr
    Fm[Fair_idx*3+2] = 5.2*Fmzl - 5.2*Fmzr
    
    # Accelerated calculation using M_inv
    force_vector = -Fz + Fw - np.dot(D, urel * np.abs(urel)) + L_cur - np.dot(K, q[udofs]) + Fm - np.dot(C, q[vdofs])
    ud = np.dot(M_inv, force_vector)
    return np.append(q[vdofs], ud)

print("Solving Non-linear FEM Time-Domain... This will be fast!")
sol = scp.solve_ivp(fun=odefun, t_span=[tspan[0], tspan[-1]], y0=q0, method='Radau', max_step=0.1)

# Plotting the main time history
fig, axs = plt.subplots(3, 1, figsize=(15, 10))
axs[0].plot(sol.t, sol.y[udofs[0]], label='Base Surge')
axs[0].plot(sol.t, sol.y[udofs[-3]], label='RNA Surge')
axs[0].legend(); axs[0].set_title('Surge')

axs[1].plot(sol.t, sol.y[udofs[1]], label='Base Heave')
axs[1].plot(sol.t, sol.y[udofs[-2]], label='RNA Heave')
axs[1].legend(); axs[1].set_title('Heave')

axs[2].plot(sol.t, np.rad2deg(sol.y[udofs[2]]), label='Base Pitch')
axs[2].plot(sol.t, np.rad2deg(sol.y[udofs[-1]]), label='RNA Pitch')
axs[2].legend(); axs[2].set_title('Pitch')
plt.tight_layout(); plt.show()

In [ ]:
# ==============================================================================
# CELL 9: MODAL ANALYSIS (REDUCED ORDER MODEL)
# ==============================================================================
mat = np.dot(M_inv, K)
w2, vr = np.linalg.eig(mat)
w_freq = np.sqrt(np.abs(w2.real))
f = w_freq / (2*np.pi)

idx_sort = f.argsort()
f = f[idx_sort]
vr_sorted = vr[:, idx_sort]

nMode = nn*3
PHI = vr_sorted[:, 0:nMode]

Mm, Km, Cm = np.zeros(nMode), np.zeros(nMode), np.zeros(nMode)
for iMode in range(nMode):
    Mm[iMode] = PHI[:,iMode].T @ M @ PHI[:,iMode]
    Km[iMode] = PHI[:,iMode].T @ K @ PHI[:,iMode]
    Cm[iMode] = 0.01*Mm[iMode] + 0.01*Km[iMode]

def qdot(t, q):
    qwave[idx] = uwave(t)
    U_F = np.sum(PHI*q[0:nMode], 1)
    V_F = np.sum(PHI*q[nMode:2*nMode], 1)
    
    urel = V_F - qwave[vdofs]
    vrel = V_F[-3] - V_hub(t)
    
    Fz[1] = np.pi*R_buoy_key[0]**2 * pbase(t)
    Fw[-3] = np.interp(vrel, thrust_table['x'], thrust_table['y'])*1e3
    
    Fmxl, Fmzl, Fmxr, Fmzr = Fmoor(U_F[6])
    Fm[Fair_idx*3] = Fmxr + Fmxl
    Fm[Fair_idx*3+1] = Fmzl + Fmzr
    Fm[Fair_idx*3+2] = 5.2*Fmzl - 5.2*Fmzr
    
    F_vec = -Fz + Fw - np.dot(D, (urel * np.abs(urel))) + L_cur + Fm
    Am = (PHI.T @ F_vec - (Km * q[0:nMode] + Cm * q[nMode:2*nMode])) / Mm
    return np.append(q[nMode:2*nMode], Am)

q0_modal = np.zeros(2*nMode)
q_sol = scp.solve_ivp(fun=qdot, t_span=[tspan[0], tspan[-1]], y0=q0_modal, method='Radau', max_step=0.1)

# Reconstruct the physical coordinates from the modal coordinates
U_F = np.zeros((len(q_sol.t), len(PHI[:,0])))
for iMode in range(nMode):
    for it in range(len(q_sol.t)):
        U_F[it,:] += PHI[:,iMode] * q_sol.y[iMode][it]

# Plot Comparison (FEM vs Modal)
fig, axs = plt.subplots(1, 2, figsize=(15, 5))
axs[0].plot(sol.t, sol.y[udofs[-3]], label='FEM RNA Surge', alpha=0.7)
axs[0].plot(q_sol.t, U_F[:, -3], '--', label='Modal RNA Surge')
axs[0].legend(); axs[0].set_title('Surge Comparison')

axs[1].plot(sol.t, np.rad2deg(sol.y[udofs[-1]]), label='FEM RNA Pitch', alpha=0.7)
axs[1].plot(q_sol.t, np.rad2deg(U_F[:, -1]), '--', label='Modal RNA Pitch')
axs[1].legend(); axs[1].set_title('Pitch Comparison')
plt.show()

In [ ]:
# ==============================================================================
# CELL 10: FREE DECAY TESTS (PITCH & HEAVE)
# ==============================================================================
def odefun_decay(t, q):             
    urel = q[vdofs] 
    Fmxl, Fmzl, Fmxr, Fmzr = Fmoor(q[udofs][6])
    Fm[Fair_idx*3] = Fmxr + Fmxl
    Fm[Fair_idx*3+1] = Fmzl + Fmzr
    Fm[Fair_idx*3+2] = 5.2*Fmzl - 5.2*Fmzr
        
    ud = np.dot(M_inv, -np.dot(D, urel*np.abs(urel)) - np.dot(K, q[udofs]) + Fm - np.dot(C, q[vdofs]))
    return np.append(q[vdofs], ud)

# --- 5 DEG PITCH DECAY ---
q0_pitch = np.zeros(2*ndofs)
du0 = np.deg2rad(5)
q0_pitch[udofs[2::3]] = du0
q0_pitch[udofs[0::3]] = du0 * zn - (du0 * zn)[0]

print("Solving Pitch Decay...")
sol_pitch = scp.solve_ivp(fun=odefun_decay, t_span=[0, 200], y0=q0_pitch, method='Radau', max_step=0.1)

# --- 1 METER HEAVE DECAY ---
q0_heave = np.zeros(2*ndofs)
q0_heave[udofs[1::3]] = 1.0

print("Solving Heave Decay...")
sol_heave = scp.solve_ivp(fun=odefun_decay, t_span=[0, 200], y0=q0_heave, method='Radau', max_step=0.1)

# Plot Results
fig, axs = plt.subplots(1, 2, figsize=(15, 5))
axs[0].plot(sol_pitch.t, np.rad2deg(sol_pitch.y[udofs[-1]]))
axs[0].set_title('5 Deg Initial Pitch Decay (RNA Node)')
axs[0].set_xlabel('Time [s]'); axs[0].set_ylabel('Pitch [deg]'); axs[0].grid()

axs[1].plot(sol_heave.t, sol_heave.y[udofs[1]])
axs[1].set_title('1 m Initial Heave Decay (Buoy Base)')
axs[1].set_xlabel('Time [s]'); axs[1].set_ylabel('Heave [m]'); axs[1].grid()
plt.show()

print(f"Total Notebook Execution Time: {time.time() - start_time:.2f} seconds")